# RAG Capstone: Generation, Citations, Grounding & Evaluation

- **Toy corpus** (10 short "company policy" style docs).
- **Local embeddings** via `sentence-transformers` (no API key needed for retrieval).
- **OpenAI** for generation (set `OPENAI_API_KEY` env var). If no key is set,
  a stub LLM is used so the rest of the pipeline still runs end-to-end for testing.
- **RAGAS** for faithfulness / relevancy / recall / precision (needs an LLM key).
- **Retrieval@K** metrics computed with pure Python (no API needed).
- A **chunking ablation** loop.
- A **deliberately broken** example (no grounding instructions) to show
  faithfulness dropping.


In [2]:
# Uncomment to install (first run only)
# %pip install sentence-transformers numpy pandas scikit-learn ragas datasets openai trulens-eval

import os
import re
import itertools
import numpy as np
import pandas as pd

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")  # set this in your shell/.env
print("OpenAI key detected:", bool(OPENAI_API_KEY))


OpenAI key detected: False


In [3]:
documents = [
    {"id": "refund_policy#p1", "text": "Standard (non-damaged) items may be returned within 14 days of delivery for a full refund."},
    {"id": "refund_policy#p2", "text": "Damaged or defective items may be returned within 30 days of delivery for a full refund or replacement."},
    {"id": "refund_policy#p3", "text": "Refunds are issued to the original payment method within 5-7 business days of the item being received."},
    {"id": "shipping_policy#p1", "text": "Standard shipping takes 3-5 business days within the continental US."},
    {"id": "shipping_policy#p2", "text": "Expedited shipping (1-2 business days) is available for an additional fee at checkout."},
    {"id": "loyalty_program#p1", "text": "Loyalty members earn 1 point per dollar spent, redeemable at 100 points = $5 off."},
    {"id": "loyalty_program#p5", "text": "Loyalty discounts cannot be combined with seasonal promotional codes; the system automatically applies whichever discount is larger."},
    {"id": "promo_terms#p1", "text": "Seasonal promo codes apply a flat 15% discount and cannot be stacked with other offers, including loyalty discounts."},
    {"id": "warranty#p1", "text": "Laptops carry a 12-month manufacturer warranty covering hardware defects."},
    {"id": "warranty#p2", "text": "Desktop computers carry a 24-month manufacturer warranty covering hardware defects."},
    {"id": "company_info#p1", "text": "The company was founded in 1998 and is headquartered in Austin, Texas."},
    {"id": "account_security#p1", "text": "Two-factor authentication (2FA) can be enabled from Account Settings > Security."},
]

print(f"{len(documents)} documents loaded")


12 documents loaded


Golden QA dataset

Covers: single-doc questions, a multi-doc synthesis question, and a
deliberately unanswerable question (tests graceful missing-context
handling).

In [4]:
golden_qa = [
    {
        "id": "gqa_001",
        "question": "What is the refund window for damaged items?",
        "ground_truth": "Damaged or defective items can be refunded within 30 days of delivery.",
        "expected_source_ids": ["refund_policy#p2"],
        "category": "policy", "difficulty": "easy",
    },
    {
        "id": "gqa_002",
        "question": "How long is the warranty on a laptop?",
        "ground_truth": "Laptops have a 12-month manufacturer warranty.",
        "expected_source_ids": ["warranty#p1"],
        "category": "warranty", "difficulty": "easy",
    },
    {
        "id": "gqa_003",
        "question": "Can I combine a loyalty discount with a seasonal promo code?",
        "ground_truth": "No. Loyalty discounts and seasonal promo codes cannot be stacked; "
                         "the system automatically applies whichever discount is larger.",
        "expected_source_ids": ["loyalty_program#p5", "promo_terms#p1"],
        "category": "pricing", "difficulty": "hard",  # multi-doc synthesis
    },
    {
        "id": "gqa_004",
        "question": "What is the CEO's name?",
        "ground_truth": "not covered",
        "expected_source_ids": [],
        "category": "unanswerable", "difficulty": "medium",  # not in corpus
    },
    {
        "id": "gqa_005",
        "question": "How can I enable two-factor authentication?",
        "ground_truth": "Enable 2FA from Account Settings > Security.",
        "expected_source_ids": ["account_security#p1"],
        "category": "account", "difficulty": "easy",
    },
]

golden_df = pd.DataFrame(golden_qa)
golden_df


,id,question,ground_truth,expected_source_ids,category,difficulty
0,gqa_001,What is the refund window for damaged items?,Damaged or defective items can be refunded wit...,[refund_policy#p2],policy,easy
1,gqa_002,How long is the warranty on a laptop?,Laptops have a 12-month manufacturer warranty.,[warranty#p1],warranty,easy
2,gqa_003,Can I combine a loyalty discount with a season...,No. Loyalty discounts and seasonal promo codes...,"[loyalty_program#p5, promo_terms#p1]",pricing,hard
3,gqa_004,What is the CEO's name?,not covered,[],unanswerable,medium
4,gqa_005,How can I enable two-factor authentication?,Enable 2FA from Account Settings > Security.,[account_security#p1],account,easy


Retrieval: local embeddings + in-memory cosine search

No API key required for this part. Chunking is deliberately parameterized
(`chunk_size`, `chunk_overlap`).

In [ ]:
from sentence_transformers import SentenceTransformer

_embedder = SentenceTransformer("all-MiniLM-L6-v2")

def simple_chunk(text: str, chunk_size: int, chunk_overlap: int) -> list[str]:
    """Naive char-based splitter -- swap for RecursiveCharacterTextSplitter if you like."""
    if chunk_overlap >= chunk_size:
        chunk_overlap = chunk_size // 2
    chunks, start = [], 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - chunk_overlap
    return [c for c in chunks if c.strip()]

def build_index(documents: list[dict], chunk_size: int, chunk_overlap: int):
    """Returns (chunk_records, chunk_embeddings) for a given chunking config."""
    records = []
    for doc in documents:
        for i, piece in enumerate(simple_chunk(doc["text"], chunk_size, chunk_overlap)):
            records.append({"id": f'{doc["id"]}::c{i}', "source_id": doc["id"], "text": piece})
    texts = [r["text"] for r in records]
    embeddings = _embedder.encode(texts, normalize_embeddings=True)
    return records, embeddings

def retrieve(question: str, records, embeddings, k: int = 3):
    q_emb = _embedder.encode([question], normalize_embeddings=True)[0]
    sims = embeddings @ q_emb  # cosine sim since both are normalized
    top_idx = np.argsort(-sims)[:k]
    return [{**records[i], "score": float(sims[i])} for i in top_idx]

# Quick smoke test with a default chunking config
records, embeddings = build_index(documents, chunk_size=200, chunk_overlap=30)
retrieve("What is the refund window for damaged items?", records, embeddings, k=3)


Generation: grounded prompt + citations + missing-context guard

This wires together Module 1 from the learning plan (prompt structure,
missing-context handling, citations, grounding). Falls back to a stub LLM
if no `OPENAI_API_KEY` is set.

In [ ]:
GROUNDED_TEMPLATE = """You are a careful assistant that answers ONLY from the provided context.

# CONTEXT
{context}

# QUESTION
{question}

# STRICT RULES
- Every sentence in your answer must be directly supported by the CONTEXT.
- Do NOT add facts, numbers, or names not present in the CONTEXT.
- Cite each claim's source using [source_id] notation.
- If the CONTEXT does not contain the answer, respond exactly:
  "The provided context does not cover this."

# ANSWER
"""

SIMILARITY_THRESHOLD = 0.30  # tune once you inspect real score distributions

def call_llm(prompt: str, temperature: float = 0.0) -> str:
    if OPENAI_API_KEY:
        from openai import OpenAI
        client = OpenAI(api_key=OPENAI_API_KEY)
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=temperature,
        )
        return resp.choices[0].message.content
    # --- stub fallback so the pipeline is testable without a key ---
    return "[stub-llm] No OPENAI_API_KEY set -- this is a placeholder answer."

def format_context(chunks: list[dict]) -> str:
    return "\n\n".join(f'[{c["source_id"]}]\n{c["text"]}' for c in chunks)

def parse_citations(answer: str) -> list[str]:
    return list(dict.fromkeys(re.findall(r"\[([\w#\.\-]+)\]", answer)))

def answer_question(question: str, records, embeddings, k: int = 3, use_grounding: bool = True):
    hits = retrieve(question, records, embeddings, k=k)
    hits = [h for h in hits if h["score"] >= SIMILARITY_THRESHOLD]

    if not hits:
        return {"answer": "The provided context does not cover this.",
                "citations": [], "unverified_citations": [], "contexts": []}

    context_str = format_context(hits)
    template = GROUNDED_TEMPLATE if use_grounding else "{context}\n\nQuestion: {question}\nAnswer:"
    prompt = template.format(context=context_str, question=question)

    raw = call_llm(prompt)
    cited = parse_citations(raw)
    valid_ids = {h["source_id"] for h in hits}
    unverified = [c for c in cited if c not in valid_ids]

    return {
        "answer": raw,
        "citations": cited,
        "unverified_citations": unverified,
        "contexts": [h["text"] for h in hits],
    }

# Try it
result = answer_question("What is the refund window for damaged items?", records, embeddings)
result


Chunking ablation study

Sweeps `chunk_size` / `chunk_overlap`, and scores each config with
**Retrieval@K** (free, no API calls) so you can iterate fast.

In [ ]:
def precision_at_k(retrieved_source_ids, relevant_ids, k):
    top_k = retrieved_source_ids[:k]
    return sum(1 for i in top_k if i in relevant_ids) / k if k else 0.0

def recall_at_k(retrieved_source_ids, relevant_ids, k):
    if not relevant_ids:
        return None  # undefined for unanswerable questions
    top_k = retrieved_source_ids[:k]
    return sum(1 for i in top_k if i in relevant_ids) / len(relevant_ids)

def reciprocal_rank(retrieved_source_ids, relevant_ids):
    for rank, sid in enumerate(retrieved_source_ids, start=1):
        if sid in relevant_ids:
            return 1.0 / rank
    return 0.0

def evaluate_retrieval_config(documents, golden_df, chunk_size, chunk_overlap, k=3):
    records, embeddings = build_index(documents, chunk_size, chunk_overlap)
    p_scores, r_scores, rr_scores = [], [], []

    for _, row in golden_df.iterrows():
        relevant = set(row["expected_source_ids"])
        if not relevant:
            continue  # skip unanswerable questions for this metric
        hits = retrieve(row["question"], records, embeddings, k=k)
        retrieved_source_ids = [h["source_id"] for h in hits]

        p_scores.append(precision_at_k(retrieved_source_ids, relevant, k))
        r_scores.append(recall_at_k(retrieved_source_ids, relevant, k))
        rr_scores.append(reciprocal_rank(retrieved_source_ids, relevant))

    return {
        "chunk_size": chunk_size, "chunk_overlap": chunk_overlap,
        f"precision@{k}": np.mean(p_scores), f"recall@{k}": np.mean(r_scores),
        "MRR": np.mean(rr_scores),
    }

chunk_sizes = [100, 200, 400]
overlaps = [0, 30]

ablation_rows = [
    evaluate_retrieval_config(documents, golden_df, size, overlap, k=3)
    for size, overlap in itertools.product(chunk_sizes, overlaps)
]

ablation_df = pd.DataFrame(ablation_rows).sort_values("recall@3", ascending=False)
ablation_df


RAGAS scorecard (needs `OPENAI_API_KEY`)

In [ ]:
if OPENAI_API_KEY:
    from ragas import evaluate
    from ragas.metrics import faithfulness, answer_relevancy, context_recall, context_precision
    from datasets import Dataset

    best_size, best_overlap = int(ablation_df.iloc[0]["chunk_size"]), int(ablation_df.iloc[0]["chunk_overlap"])
    records, embeddings = build_index(documents, best_size, best_overlap)

    eval_rows = []
    for _, row in golden_df.iterrows():
        r = answer_question(row["question"], records, embeddings, k=3)
        eval_rows.append({
            "question": row["question"],
            "answer": r["answer"],
            "contexts": r["contexts"] if r["contexts"] else [""],
            "ground_truth": row["ground_truth"],
        })

    ragas_dataset = Dataset.from_list(eval_rows)
    ragas_results = evaluate(
        ragas_dataset,
        metrics=[faithfulness, answer_relevancy, context_recall, context_precision],
    )
    print(ragas_results)
else:
    print("Skipping RAGAS scorecard -- set OPENAI_API_KEY to run this cell.")


Proving grounding matters: a deliberately broken run

Same question, same retrieved context, but with grounding instructions
stripped out (`use_grounding=False`).

In [ ]:
records, embeddings = build_index(documents, chunk_size=200, chunk_overlap=30)

grounded_result = answer_question(
    "Can I combine a loyalty discount with a seasonal promo code?",
    records, embeddings, k=3, use_grounding=True,
)
ungrounded_result = answer_question(
    "Can I combine a loyalty discount with a seasonal promo code?",
    records, embeddings, k=3, use_grounding=False,
)

print("GROUNDED:\n", grounded_result["answer"])
print("\nUNGROUNDED:\n", ungrounded_result["answer"])

if OPENAI_API_KEY:
    from ragas import evaluate
    from ragas.metrics import faithfulness
    from datasets import Dataset

    compare_rows = [
        {"question": "Can I combine a loyalty discount with a seasonal promo code?",
         "answer": grounded_result["answer"], "contexts": grounded_result["contexts"]},
        {"question": "Can I combine a loyalty discount with a seasonal promo code?",
         "answer": ungrounded_result["answer"], "contexts": ungrounded_result["contexts"]},
    ]
    scores = evaluate(Dataset.from_list(compare_rows), metrics=[faithfulness])
    print(scores)


TruLens tracing (optional, for live/interactive debugging)

TruLens instruments live calls and
gives you a dashboard to click into any single question and see exactly
which step -- retrieval or generation -- caused a low score.

In [ ]:
if OPENAI_API_KEY:
    from trulens_eval import Feedback, Tru, Select
    from trulens_eval.feedback.provider import OpenAI as TruOpenAI

    tru = Tru()
    provider = TruOpenAI()

    f_groundedness = Feedback(provider.groundedness_measure_with_cot_reasons).on_output()
    f_answer_relevance = Feedback(provider.relevance).on_input_output()

    # Wrap `answer_question` as a simple TruLens "app" via the basic app wrapper
    from trulens_eval.tru_basic_app import TruBasicApp

    def rag_app(question: str) -> str:
        return answer_question(question, records, embeddings, k=3)["answer"]

    tru_app = TruBasicApp(rag_app, app_id="rag_v1", feedbacks=[f_groundedness, f_answer_relevance])

    with tru_app as recording:
        rag_app("Can I combine a loyalty discount with a seasonal promo code?")

    print("Run tru.run_dashboard() in a terminal (not inline) to open the UI.")
else:
    print("Skipping TruLens -- set OPENAI_API_KEY to run this cell.")
